# Lumis 06: Remove non-skin images

Grad-CAM showed that DermNet's scabies folder includes photos of insects (a spider, a tick) rather than skin. This notebook finds and removes images like that from every class.

1. Measures the share of skin-colored pixels in every image (a simple color rule)
2. Flags images with little skin and shows them in numbered grids, least skin first
3. **You decide** which flagged images to remove by typing their numbers
4. Saves `lumis_split_v2.csv`: the v1 split minus the removed images. Every remaining image keeps its train/val/test assignment

**Be careful with dark skin.** A color rule can under-count some skin tones, lighting or very red rashes, so a real skin photo can get flagged. Only remove images that are clearly not a photo of skin: insects, objects, diagrams, slides of text, microscope images. When in doubt, keep it. Wrongly removing dark-skin photos would make the fairness problem worse.

Runtime: CPU is fine.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

BASE = '/content/drive/MyDrive/Lumis'
LOCAL = '/content/dermnet'
V1 = f'{BASE}/data/lumis_split_v1.csv'
V2 = f'{BASE}/data/lumis_split_v2.csv'
if os.path.exists(V2):
    raise RuntimeError(f'{V2} already exists. Rename it first if you really want to redo the review.')
if not os.path.exists(LOCAL):
    with zipfile.ZipFile(f'{BASE}/data/dermnet.zip') as z:
        z.extractall(LOCAL)
df = pd.read_csv(V1)
print(len(df), 'images in split v1')

## 2. Measure skin-pixel share

A pixel counts as skin-colored if its YCbCr chroma falls in a broad skin range (Cr 130 to 190, Cb 70 to 130). The range is deliberately wide so red rashes and darker skin count as skin. Takes 2 to 3 minutes.

In [ ]:
def skin_fraction(path):
    with Image.open(path) as im:
        ycc = np.asarray(im.convert('RGB').resize((128, 128)).convert('YCbCr'), dtype=np.int16)
    cb, cr = ycc[..., 1], ycc[..., 2]
    return float(((cr >= 130) & (cr <= 190) & (cb >= 70) & (cb <= 130)).mean())

df['skin_frac'] = [skin_fraction(Path(LOCAL) / p) for p in tqdm(df.path)]

fig, ax = plt.subplots(figsize=(8, 3.5))
for cls, sub in df.groupby('lumis_class'):
    ax.hist(sub.skin_frac, bins=40, alpha=0.5, label=cls)
ax.set_xlabel('share of skin-colored pixels'); ax.set_ylabel('images'); ax.legend()
plt.show()

## 3. Flag low-skin images

Raise `THRESHOLD` to catch more, lower it to review fewer. 0.25 is a reasonable start.

In [ ]:
THRESHOLD = 0.25
flagged = df[df.skin_frac < THRESHOLD].sort_values('skin_frac').reset_index()
flagged['num'] = range(len(flagged))
print(len(flagged), 'images flagged for review')
print(flagged.lumis_class.value_counts())

## 4. Review the flagged images

Each tile shows its **number**, class, and skin share. Note the numbers of images that are clearly **not** skin photos.

In [ ]:
PER_PAGE = 40
for start in range(0, len(flagged), PER_PAGE):
    page = flagged.iloc[start:start + PER_PAGE]
    rows = int(np.ceil(len(page) / 8))
    fig, axes = plt.subplots(rows, 8, figsize=(20, 2.9 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, r) in zip(axes.flat, page.iterrows()):
        ax.imshow(Image.open(Path(LOCAL) / r.path).convert('RGB'))
        ax.set_title(f'#{r.num}  {r.lumis_class}  ({r.skin_frac:.0%})', fontsize=10)
    fig.suptitle(f'Flagged images {start} to {start + len(page) - 1}', fontsize=13)
    plt.tight_layout()
    plt.show()

### 4b. Save the review pages to Drive (optional)

Saves the same numbered grids as images in `Lumis/exports/review_flagged/`, plus a CSV of the flagged images, so they can be reviewed outside Colab.

In [ ]:
os.makedirs(f'{BASE}/exports/review_flagged', exist_ok=True)
for start in range(0, len(flagged), PER_PAGE):
    page = flagged.iloc[start:start + PER_PAGE]
    rows = int(np.ceil(len(page) / 8))
    fig, axes = plt.subplots(rows, 8, figsize=(20, 2.9 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, r) in zip(axes.flat, page.iterrows()):
        ax.imshow(Image.open(Path(LOCAL) / r.path).convert('RGB'))
        ax.set_title(f'#{r.num}  {r.lumis_class}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{BASE}/exports/review_flagged/page_{start // PER_PAGE:02d}.jpg', dpi=60)
    plt.close()
flagged[['num', 'path', 'lumis_class', 'split', 'skin_frac']].to_csv(f'{BASE}/exports/review_flagged/flagged.csv', index=False)
print('Saved', int(np.ceil(len(flagged) / PER_PAGE)), 'pages to Lumis/exports/review_flagged')

## 5. List the images to remove

Type the numbers of the non-skin images, separated by commas. Ranges work too: `'3, 7, 12-15'`.

To double-check before saving, run the next cell: it shows only the images you picked.

In [ ]:
REMOVE = ''   # e.g. '0, 1, 2, 5, 9-12'

def parse(spec):
    out = set()
    for part in spec.replace(' ', '').split(','):
        if not part:
            continue
        if '-' in part:
            a, b = part.split('-')
            out.update(range(int(a), int(b) + 1))
        else:
            out.add(int(part))
    return sorted(out)

picked = flagged[flagged.num.isin(parse(REMOVE))]
print(len(picked), 'images picked for removal')
print(picked.lumis_class.value_counts())

if len(picked):
    rows = int(np.ceil(len(picked) / 8))
    fig, axes = plt.subplots(rows, 8, figsize=(20, 2.9 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, r) in zip(axes.flat, picked.iterrows()):
        ax.imshow(Image.open(Path(LOCAL) / r.path).convert('RGB'))
        ax.set_title(f'#{r.num}  {r.lumis_class}', fontsize=10)
    fig.suptitle('These will be removed', fontsize=13)
    plt.tight_layout()
    plt.show()

## 6. Save split v2

In [ ]:
assert len(picked), 'Nothing picked: fill in REMOVE above first'
removed = picked[['path', 'lumis_class', 'split', 'skin_frac']].assign(reason='not a skin photo (manual review)')
removed.to_csv(f'{BASE}/exports/removed_nonskin_v2.csv', index=False)

v2 = df[~df.path.isin(removed.path)].drop(columns=['skin_frac'])
v2.to_csv(V2, index=False)

before = pd.crosstab(df.lumis_class, df.split)[['train', 'val', 'test']]
after = pd.crosstab(v2.lumis_class, v2.split, margins=True, margins_name='total')[['train', 'val', 'test', 'total']]
after.to_csv(f'{BASE}/exports/split_counts_v2.csv')
print('Removed by class and split:')
display(before - pd.crosstab(v2.lumis_class, v2.split)[['train', 'val', 'test']])
print('Split v2:')
display(after)
print('Saved', V2)

## 7. Second pass: search file names

Pink-stained microscope slides can look skin-colored, so the color rule misses them. DermNet names many of them (`...-histology-...`, `...-laboratory-...`) and its insect photos (`spider-...`, `head-lice-...`), so this searches the file names of every image still in split v2 and shows the matches for review, labeled `k0`, `k1`, and so on.

In [ ]:
v2 = pd.read_csv(V2)

def show(rows_df, title, label):
    for start in range(0, len(rows_df), 40):
        page = rows_df.iloc[start:start + 40]
        rr = int(np.ceil(len(page) / 8))
        fig, axes = plt.subplots(rr, 8, figsize=(20, 2.9 * rr), squeeze=False)
        for ax in axes.flat:
            ax.axis('off')
        for ax, (_, x) in zip(axes.flat, page.iterrows()):
            ax.imshow(Image.open(Path(LOCAL) / x.path).convert('RGB'))
            ax.set_title(label(x), fontsize=9)
        fig.suptitle(f'{title}: {start} to {start + len(page) - 1}', fontsize=13)
        plt.tight_layout()
        plt.show()

pattern = r'histolog|patholog|laborator|microscop|biops|culture|koh|lice|spider|flea|fire-ant|ants-|mite|forest|map'
suspect = v2[v2.path.str.lower().str.contains(pattern, regex=True)].reset_index(drop=True)
suspect['k'] = range(len(suspect))
print(len(suspect), 'images in split v2 have a suspicious file name')
show(suspect, 'Suspicious file names', lambda x: f'k{x.k}  {Path(x.path).stem[:22]}')

Type the `k` numbers of the non-skin images. The values below are the ones picked in the original review (2026-11-09).

In [ ]:
EXTRA = 'k0-k4, k12, k25-k85, k101, k123-k125, k135, k153, k156, k160, k169'

def parse_k(spec):
    out = set()
    for part in spec.replace(' ', '').replace('k', '').split(','):
        if not part:
            continue
        if '-' in part:
            a, b = part.split('-')
            out.update(range(int(a), int(b) + 1))
        else:
            out.add(int(part))
    return sorted(out)

extra = suspect[suspect.k.isin(parse_k(EXTRA))]
print(len(extra), 'more images picked for removal')
print(extra.lumis_class.value_counts())
show(extra, 'These will also be removed', lambda x: f'k{x.k}  {Path(x.path).stem[:22]}')

In [ ]:
removed_all = pd.concat([
    pd.read_csv(f'{BASE}/exports/removed_nonskin_v2.csv'),
    extra[['path', 'lumis_class', 'split']].assign(skin_frac=np.nan, reason='not a skin photo (file-name search + manual review)'),
]).drop_duplicates('path')
removed_all.to_csv(f'{BASE}/exports/removed_nonskin_v2.csv', index=False)

v2 = v2[~v2.path.isin(extra.path)]
v2.to_csv(V2, index=False)
after = pd.crosstab(v2.lumis_class, v2.split, margins=True, margins_name='total')[['train', 'val', 'test', 'total']]
after.to_csv(f'{BASE}/exports/split_counts_v2.csv')
print(len(removed_all), 'images removed in total; split v2 now has', len(v2))
display(after)